# Deep-Sight — YOLO11s training (Colab), box-fix run

Trains `backend/detect/train.py` on a Colab GPU. Primary model **YOLO11s**, run name
`deepsight_y11s_boxfix` (fresh run — does not resume the earlier `deepsight_y11s`).
Checkpoints every epoch to Google Drive — a 12 h disconnect costs nothing: reconnect,
re-run cells 2–4, then the Train cell, and it resumes from `last.pt`.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**.
2. In Google Drive `MyDrive/deepsight/`, replace both files with the new ones:
   - `deepsight_code.zip`  (from `C:\\projects\\deepsight\\deepsight_code.zip`)
   - `yolo.zip`            (from `C:\\projects\\deepsight\\data\\detect\\yolo.zip`, ~887 MB)

   Wait for both to reach 100%.
3. Runtime → **Run all**.

Output: `MyDrive/deepsight/runs/deepsight_y11s_boxfix/weights/best.pt`, also copied to
`MyDrive/deepsight/best.pt` by cell 7. Download it to
`backend/detect/weights/best.pt` in the repo and commit.

In [ ]:
!nvidia-smi
!pip -q install ultralytics

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/deepsight'
os.makedirs(DRIVE, exist_ok=True)
print('contents of', DRIVE, '->', os.listdir(DRIVE))
assert 'yolo.zip' in os.listdir(DRIVE), 'upload yolo.zip to MyDrive/deepsight/ first'
assert 'deepsight_code.zip' in os.listdir(DRIVE), 'upload deepsight_code.zip to MyDrive/deepsight/ first'

In [ ]:
# --- get the code ---
%cd /content
!rm -rf deepsight && mkdir deepsight
REPO_URL = ''  # optional: 'https://github.com/<you>/deepsight.git'. Empty -> use deepsight_code.zip
if REPO_URL:
    !git clone $REPO_URL deepsight
else:
    !cd deepsight && unzip -q -o $DRIVE/deepsight_code.zip
%cd /content/deepsight
!touch backend/__init__.py scripts/__init__.py
!python -c "import backend.detect.dataset, backend.detect.train; print('code import OK')"

In [ ]:
# --- unzip the dataset and repoint data.yaml at the Colab path ---
%cd /content/deepsight
!mkdir -p data/detect
!cd data/detect && unzip -q -o $DRIVE/yolo.zip && echo unzipped
import pathlib
COLAB_DATA = '/content/deepsight/data/detect/yolo'
y = pathlib.Path('data/detect/yolo/data.yaml')
keep = [ln for ln in y.read_text().splitlines() if not ln.startswith('path:')]
y.write_text('path: ' + COLAB_DATA + '\n' + '\n'.join(keep) + '\n')
print(y.read_text())
n_tr = len(list(pathlib.Path(COLAB_DATA, 'images/train').glob('*.jpg')))
n_va = len(list(pathlib.Path(COLAB_DATA, 'images/val').glob('*.jpg')))
print('images  train:', n_tr, ' val:', n_va)
assert n_tr > 0 and n_va > 0, 'dataset did not unzip correctly'

In [ ]:
# --- TRAIN: YOLO11s, box-fix run. Re-run after any disconnect; it resumes from last.pt ---
%cd /content/deepsight
!python -m backend.detect.train --data data/detect/yolo/data.yaml --model yolo11s.pt --epochs 100 --imgsz 640 --batch 16 --device 0 --project $DRIVE/runs --name deepsight_y11s_boxfix

In [ ]:
# --- results + export best.pt ---
import os, shutil
from IPython.display import Image, display
run = DRIVE + '/runs/deepsight_y11s_boxfix'
shutil.copy(run + '/weights/best.pt', DRIVE + '/best.pt')
print('copied ->', DRIVE + '/best.pt')
print('download it, place at  backend/detect/weights/best.pt  in the repo, commit it')
for p in ('results.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg'):
    fp = run + '/' + p
    if os.path.exists(fp):
        display(Image(fp))

In [ ]:
# --- OPTIONAL BASELINE: YOLOv8s box-fix run, for the cross-model eval comparison.
#     Adds ~1 h. Skip it if you only need the primary model. ---
%cd /content/deepsight
!python -m backend.detect.train --data data/detect/yolo/data.yaml --model yolov8s.pt --epochs 100 --imgsz 640 --batch 16 --device 0 --project $DRIVE/runs --name deepsight_y8s_boxfix